# Weather Data Validation — PJM Observed Temps

**Source:** `dbt_wsi_temps_v1_2026_feb_25.source_v1_hourly_observed_temp`  
**Station:** PJM population-weighted aggregate  
**Purpose:** Validate raw hourly weather data before it enters the like-day feature pipeline.

### Checks
1. **Completeness** — Date range, hourly coverage, missing days
2. **Nulls & Outliers** — Column-level null counts, statistical outlier detection
3. **Distributions** — Histograms and seasonal patterns for key fields
4. **Temporal Continuity** — Day-over-day jumps, gap detection
5. **HDD/CDD Sanity** — Verify derived features against raw temps
6. **Cross-Validation vs Load** — Temperature-load correlation check

## 1. Setup & Data Pull

In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

import pjm_like_day_forecast.settings
from pjm_like_day_forecast.data import weather_hourly
from pjm_like_day_forecast import configs

HOURS = list(range(1, 25))

In [2]:
# Pull raw hourly data for PJM aggregate station
df_raw = weather_hourly.pull()
df_raw["date"] = pd.to_datetime(df_raw["date"])

print(f"Shape: {df_raw.shape}")
print(f"Date range: {df_raw['date'].min().date()} to {df_raw['date'].max().date()}")
print(f"Unique dates: {df_raw['date'].nunique():,}")
print(f"\nColumns: {list(df_raw.columns)}")
print(f"\nDtypes:\n{df_raw.dtypes}")
df_raw.head()

2026-02-25 18:58:29 | ℹ️  INFO | weather_hourly.py:pull:23 | Pulling observed weather hourly: PJM from dbt_wsi_temps_v1_2026_feb_25
2026-02-25 18:59:11 | ℹ️  INFO | weather_hourly.py:pull:27 | Pulled 271,673 rows
Shape: (271673, 9)
Date range: 1995-01-01 to 2026-02-25
Unique dates: 11,320

Columns: ['date', 'hour_ending', 'station_name', 'temp', 'feels_like_temp', 'dew_point_temp', 'wind_speed_mph', 'relative_humidity', 'cloud_cover_pct']

Dtypes:
date                 datetime64[s]
hour_ending                  int64
station_name                   str
temp                         int64
feels_like_temp              int64
dew_point_temp               int64
wind_speed_mph               int64
relative_humidity            int64
cloud_cover_pct            float64
dtype: object


,date,hour_ending,station_name,temp,feels_like_temp,dew_point_temp,wind_speed_mph,relative_humidity,cloud_cover_pct
0,1995-01-01,1,PJM,38,38,37,7,95,100.0
1,1995-01-01,2,PJM,38,38,37,8,96,100.0
2,1995-01-01,3,PJM,38,38,37,7,95,100.0
3,1995-01-01,4,PJM,38,38,37,7,96,100.0
4,1995-01-01,5,PJM,38,38,37,7,95,100.0


### Previous 3 Days — All Weather Components

Quick visual inspection of the most recent 3 days across all weather variables: temperature, feels-like, dew point, wind speed, humidity, and cloud cover.

In [ ]:
# Previous 3 days — all weather components
last_3_dates = sorted(df_raw["date"].unique())[-3:]
df_last3 = df_raw[df_raw["date"].isin(last_3_dates)].sort_values(["date", "hour_ending"])
print(f"Previous 3 days: {[str(d.date()) for d in last_3_dates]}")
print(f"Rows: {len(df_last3)}\n")

print("=== Temperature (°F) — Previous 3 Days ===")
display(df_last3.pivot_table(index="hour_ending", columns="date", values="temp"))

print("\n=== Feels-Like Temperature (°F) — Previous 3 Days ===")
display(df_last3.pivot_table(index="hour_ending", columns="date", values="feels_like_temp"))

print("\n=== Dew Point Temperature (°F) — Previous 3 Days ===")
display(df_last3.pivot_table(index="hour_ending", columns="date", values="dew_point_temp"))

print("\n=== Wind Speed (mph) — Previous 3 Days ===")
display(df_last3.pivot_table(index="hour_ending", columns="date", values="wind_speed_mph"))

print("\n=== Relative Humidity (%) — Previous 3 Days ===")
display(df_last3.pivot_table(index="hour_ending", columns="date", values="relative_humidity"))

print("\n=== Cloud Cover (%) — Previous 3 Days ===")
display(df_last3.pivot_table(index="hour_ending", columns="date", values="cloud_cover_pct"))

## 2. Completeness Checks

Verify every date has 24 hourly observations and identify any gaps in the date sequence.

In [ ]:
# 2a. Hours per day — flag any day with != 24 hours
hours_per_day = df_raw.groupby("date")["hour_ending"].nunique().rename("n_hours")
incomplete = hours_per_day[hours_per_day != 24]

print(f"Total dates: {len(hours_per_day):,}")
print(f"Complete (24h): {(hours_per_day == 24).sum():,}")
print(f"Incomplete: {len(incomplete)}")

if len(incomplete) > 0:
    print(f"\nIncomplete days (showing up to 20):")
    display(incomplete.tail(20).to_frame())

# 2b. Date sequence gaps
all_dates = pd.date_range(df_raw["date"].min(), df_raw["date"].max(), freq="D")
present_dates = set(df_raw["date"].dt.normalize().unique())
missing_dates = sorted(set(all_dates) - present_dates)

print(f"\nExpected dates in range: {len(all_dates):,}")
print(f"Present dates: {len(present_dates):,}")
print(f"Missing dates: {len(missing_dates)}")

if 0 < len(missing_dates) <= 30:
    print("Missing:")
    for d in missing_dates:
        print(f"  {d.date()}")
elif len(missing_dates) > 30:
    print(f"First 10 missing: {[d.date() for d in missing_dates[:10]]}")
    print(f"Last 10 missing:  {[d.date() for d in missing_dates[-10:]]}")

# 2c. Duplicates
dupes = df_raw.duplicated(subset=["date", "hour_ending"], keep=False)
print(f"\nDuplicate (date, hour_ending) rows: {dupes.sum()}")

## 3. Null & Outlier Analysis

In [ ]:
numeric_cols = ["temp", "feels_like_temp", "dew_point_temp", "wind_speed_mph",
                "relative_humidity", "cloud_cover_pct"]

# Null summary
null_summary = pd.DataFrame({
    "null_count": df_raw[numeric_cols].isnull().sum(),
    "null_pct": (df_raw[numeric_cols].isnull().sum() / len(df_raw) * 100).round(3),
})
print("Null Summary:")
display(null_summary)

# Descriptive statistics
print("\nDescriptive Statistics:")
display(df_raw[numeric_cols].describe().round(2))

# Outlier detection (|z| > 4)
print("\nOutlier Check (|z| > 4):")
for col in numeric_cols:
    series = df_raw[col].dropna()
    z = (series - series.mean()) / series.std()
    n_outliers = (z.abs() > 4).sum()
    print(f"  {col}: {n_outliers} outliers  (range [{series.min()}, {series.max()}])")

# Physical plausibility
print("\n--- Physical Plausibility ---")
print(f"  temp < -30F:     {(df_raw['temp'] < -30).sum()}")
print(f"  temp > 120F:     {(df_raw['temp'] > 120).sum()}")
print(f"  wind < 0 mph:    {(df_raw['wind_speed_mph'] < 0).sum()}")
print(f"  wind > 80 mph:   {(df_raw['wind_speed_mph'] > 80).sum()}")
print(f"  humidity < 0%:   {(df_raw['relative_humidity'] < 0).sum()}")
print(f"  humidity > 100%: {(df_raw['relative_humidity'] > 100).sum()}")

## 4. Distributions

In [ ]:
# Histograms for all numeric columns
fig = make_subplots(rows=2, cols=3, subplot_titles=numeric_cols)
for i, col in enumerate(numeric_cols):
    row, col_idx = divmod(i, 3)
    fig.add_trace(
        go.Histogram(x=df_raw[col], nbinsx=80, name=col, showlegend=False),
        row=row + 1, col=col_idx + 1,
    )
fig.update_layout(height=600, title_text="Hourly Weather Variable Distributions — PJM Aggregate")
fig.show()

In [ ]:
# Monthly temperature boxplot — seasonal sanity check
df_raw["month"] = df_raw["date"].dt.month
month_names = {1:"Jan",2:"Feb",3:"Mar",4:"Apr",5:"May",6:"Jun",
               7:"Jul",8:"Aug",9:"Sep",10:"Oct",11:"Nov",12:"Dec"}
df_raw["month_name"] = df_raw["month"].map(month_names)

fig = px.box(df_raw, x="month_name", y="temp",
             category_orders={"month_name": list(month_names.values())},
             title="Monthly Temperature Distribution — PJM Aggregate (all years)",
             labels={"temp": "Temperature (°F)", "month_name": "Month"})
fig.add_hline(y=65, line_dash="dot", line_color="gray", annotation_text="65°F (HDD/CDD base)")
fig.show()

In [ ]:
# Diurnal temperature profile by season
df_raw["season"] = df_raw["month"].map(
    {12:"Winter",1:"Winter",2:"Winter",3:"Spring",4:"Spring",5:"Spring",
     6:"Summer",7:"Summer",8:"Summer",9:"Fall",10:"Fall",11:"Fall"})

diurnal = df_raw.groupby(["season", "hour_ending"])["temp"].mean().reset_index()

fig = px.line(diurnal, x="hour_ending", y="temp", color="season",
              category_orders={"season": ["Winter", "Spring", "Summer", "Fall"]},
              markers=True,
              title="Average Diurnal Temperature Profile by Season — PJM Aggregate",
              labels={"temp": "Mean Temperature (°F)", "hour_ending": "Hour Ending"})
fig.update_layout(xaxis=dict(dtick=1))
fig.show()

## 5. Temporal Continuity

In [ ]:
# Daily average temp time series + day-over-day change
daily_avg = df_raw.groupby("date")["temp"].mean().rename("temp_daily_avg")
daily_avg = daily_avg.sort_index()
daily_change = daily_avg.diff().rename("temp_daily_change")

fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                    subplot_titles=["PJM Daily Average Temperature", "Day-over-Day Change"])

fig.add_trace(go.Scatter(x=daily_avg.index, y=daily_avg.values, mode="lines",
                          line=dict(width=0.5, color="steelblue"), name="Daily Avg"),
              row=1, col=1)
fig.add_hline(y=65, line_dash="dot", line_color="gray", row=1, col=1)

fig.add_trace(go.Scatter(x=daily_change.index, y=daily_change.values, mode="lines",
                          line=dict(width=0.5, color="steelblue"), name="Daily Change"),
              row=2, col=1)

# Flag extreme jumps
extreme_mask = daily_change.abs() > 25
if extreme_mask.sum() > 0:
    fig.add_trace(go.Scatter(x=daily_change.index[extreme_mask], y=daily_change.values[extreme_mask],
                              mode="markers", marker=dict(color="red", size=6),
                              name=f"Extreme (>25°F): {extreme_mask.sum()}"),
                  row=2, col=1)

fig.update_layout(height=600, showlegend=True)
fig.update_yaxes(title_text="Temp (°F)", row=1, col=1)
fig.update_yaxes(title_text="Change (°F)", row=2, col=1)
fig.show()

print(f"Day-over-day change: mean={daily_change.mean():.2f}°F, std={daily_change.std():.2f}°F")
print(f"  Min: {daily_change.min():.1f}°F on {daily_change.idxmin().date()}")
print(f"  Max: {daily_change.max():.1f}°F on {daily_change.idxmax().date()}")
print(f"  |change| > 20°F: {(daily_change.abs() > 20).sum()} days")
print(f"  |change| > 25°F: {(daily_change.abs() > 25).sum()} days")

In [ ]:
# Hour-over-hour jump analysis
df_sorted = df_raw.sort_values(["date", "hour_ending"]).copy()
df_sorted["temp_hourly_change"] = df_sorted.groupby("date")["temp"].diff()

hourly_extreme = df_sorted["temp_hourly_change"].abs() > 15
print(f"Hour-over-hour |change| > 15°F: {hourly_extreme.sum()} observations")
if hourly_extreme.sum() > 0:
    display(df_sorted.loc[hourly_extreme, ["date", "hour_ending", "temp", "temp_hourly_change"]].head(20))

fig = px.histogram(df_sorted, x="temp_hourly_change", nbins=60,
                   title="Distribution of Hour-over-Hour Temperature Changes",
                   labels={"temp_hourly_change": "Hour-over-Hour Temp Change (°F)"})
fig.show()

## 6. HDD/CDD Feature Sanity Check

In [ ]:
from pjm_like_day_forecast.features import weather_features

df_feat = weather_features.build(df_raw)
df_feat["date"] = pd.to_datetime(df_feat["date"])
print(f"Weather features shape: {df_feat.shape}")
print(f"Columns: {list(df_feat.columns)}")
display(df_feat.describe().round(2))

In [ ]:
# Verify HDD/CDD formulas
df_check = df_feat[["date", "temp_daily_avg", "hdd", "cdd"]].copy()
df_check["hdd_expected"] = np.maximum(0, 65 - df_check["temp_daily_avg"])
df_check["cdd_expected"] = np.maximum(0, df_check["temp_daily_avg"] - 65)
df_check["hdd_match"] = np.isclose(df_check["hdd"], df_check["hdd_expected"], atol=0.01)
df_check["cdd_match"] = np.isclose(df_check["cdd"], df_check["cdd_expected"], atol=0.01)
df_check["both_positive"] = (df_check["hdd"] > 0) & (df_check["cdd"] > 0)

print("HDD/CDD Identity Checks:")
print(f"  HDD matches formula: {df_check['hdd_match'].all()}")
print(f"  CDD matches formula: {df_check['cdd_match'].all()}")
print(f"  Days with both HDD>0 and CDD>0: {df_check['both_positive'].sum()} (should be 0)")

# HDD/CDD vs temperature scatter
fig = make_subplots(rows=1, cols=2, subplot_titles=["HDD vs Daily Avg Temp", "CDD vs Daily Avg Temp"])
fig.add_trace(go.Scatter(x=df_check["temp_daily_avg"], y=df_check["hdd"],
                          mode="markers", marker=dict(size=2, opacity=0.3, color="steelblue"),
                          name="HDD"), row=1, col=1)
fig.add_trace(go.Scatter(x=df_check["temp_daily_avg"], y=df_check["cdd"],
                          mode="markers", marker=dict(size=2, opacity=0.3, color="firebrick"),
                          name="CDD"), row=1, col=2)
fig.add_vline(x=65, line_dash="dot", line_color="gray", row=1, col=1)
fig.add_vline(x=65, line_dash="dot", line_color="gray", row=1, col=2)
fig.update_xaxes(title_text="Daily Avg Temp (°F)")
fig.update_layout(height=400)
fig.show()

In [ ]:
# Annual HDD/CDD totals
df_feat["year"] = df_feat["date"].dt.year
annual = df_feat.groupby("year").agg(
    total_hdd=("hdd", "sum"), total_cdd=("cdd", "sum"), n_days=("date", "count"),
).reset_index()
annual = annual[annual["n_days"] >= 360]

fig = go.Figure()
fig.add_trace(go.Bar(x=annual["year"], y=annual["total_hdd"], name="HDD", marker_color="steelblue"))
fig.add_trace(go.Bar(x=annual["year"], y=annual["total_cdd"], name="CDD", marker_color="firebrick"))
fig.update_layout(barmode="group", title="Annual HDD and CDD Totals — PJM Aggregate",
                  xaxis_title="Year", yaxis_title="Degree Days", height=450)
fig.show()

print(f"HDD range: {annual['total_hdd'].min():.0f} - {annual['total_hdd'].max():.0f}")
print(f"CDD range: {annual['total_cdd'].min():.0f} - {annual['total_cdd'].max():.0f}")

## 7. Cross-Validation vs Load

In [ ]:
from pjm_like_day_forecast.data import load_rt_metered_hourly

df_load = load_rt_metered_hourly.pull()
df_load["date"] = pd.to_datetime(df_load["date"])
daily_load = df_load.groupby("date")["rt_load_mw"].mean().rename("load_daily_avg")

df_cross = df_feat.set_index("date").join(daily_load, how="inner")
print(f"Cross-validation dataset: {len(df_cross):,} days with both weather + load")

# Temperature vs Load scatter
fig = make_subplots(rows=1, cols=3,
                    subplot_titles=["Temp vs Load (All Days)", "HDD vs Load (Heating)", "CDD vs Load (Cooling)"])

fig.add_trace(go.Scatter(x=df_cross["temp_daily_avg"], y=df_cross["load_daily_avg"],
                          mode="markers", marker=dict(size=2, opacity=0.2, color="steelblue"),
                          name="All"), row=1, col=1)

winter = df_cross[df_cross["hdd"] > 0]
corr_hdd = winter[["hdd", "load_daily_avg"]].corr().iloc[0, 1]
fig.add_trace(go.Scatter(x=winter["hdd"], y=winter["load_daily_avg"],
                          mode="markers", marker=dict(size=2, opacity=0.3, color="steelblue"),
                          name=f"HDD (r={corr_hdd:.3f})"), row=1, col=2)

summer = df_cross[df_cross["cdd"] > 0]
corr_cdd = summer[["cdd", "load_daily_avg"]].corr().iloc[0, 1]
fig.add_trace(go.Scatter(x=summer["cdd"], y=summer["load_daily_avg"],
                          mode="markers", marker=dict(size=2, opacity=0.3, color="firebrick"),
                          name=f"CDD (r={corr_cdd:.3f})"), row=1, col=3)

fig.update_layout(height=400, title_text="Temperature-Load Cross-Validation")
fig.update_xaxes(title_text="Daily Avg Temp (°F)", row=1, col=1)
fig.update_xaxes(title_text="HDD", row=1, col=2)
fig.update_xaxes(title_text="CDD", row=1, col=3)
fig.update_yaxes(title_text="Daily Avg Load (MW)", row=1, col=1)
fig.show()

print(f"HDD vs Load (heating days): r = {corr_hdd:.3f}")
print(f"CDD vs Load (cooling days): r = {corr_cdd:.3f}")

## 8. Feature Correlation Matrix

In [ ]:
feat_cols = [c for c in df_feat.columns if c not in ("date", "year")]
corr = df_feat[feat_cols].corr()

fig = px.imshow(corr, text_auto=".2f", color_continuous_scale="RdBu_r", zmin=-1, zmax=1,
                title="Weather Feature Correlation Matrix", aspect="auto")
fig.update_layout(height=700, width=800)
fig.show()

# Flag highly correlated pairs
print("Highly correlated feature pairs (|r| > 0.95):")
for i in range(len(feat_cols)):
    for j in range(i + 1, len(feat_cols)):
        r = corr.iloc[i, j]
        if abs(r) > 0.95:
            print(f"  {feat_cols[i]} <-> {feat_cols[j]}: r = {r:.3f}")

## 9. Recent Data Spot Check

In [ ]:
# Recent 7-day daily summary
recent = df_feat.sort_values("date").tail(7)[
    ["date", "temp_daily_avg", "temp_daily_max", "temp_daily_min",
     "hdd", "cdd", "feels_like_daily_avg", "wind_speed_daily_avg",
     "humidity_daily_avg", "cloud_cover_daily_avg", "temp_daily_change"]
].copy()
recent["date"] = recent["date"].dt.strftime("%Y-%m-%d (%a)")
print("Recent 7-Day Weather Summary — PJM Aggregate")
display(recent)

In [ ]:
# Recent 3-day hourly profile
last_3_dates = sorted(df_raw["date"].unique())[-3:]
df_recent = df_raw[df_raw["date"].isin(last_3_dates)].copy()
df_recent["date_str"] = df_recent["date"].dt.strftime("%Y-%m-%d (%a)")

fig = px.line(df_recent, x="hour_ending", y="temp", color="date_str", markers=True,
              title="Recent Hourly Temperature Profiles — PJM Aggregate",
              labels={"temp": "Temperature (°F)", "hour_ending": "Hour Ending", "date_str": "Date"})
fig.update_layout(xaxis=dict(dtick=1))
fig.show()

## 10. Validation Summary

In [ ]:
n_total = len(df_raw)
n_dates = df_raw["date"].nunique()
n_complete = (hours_per_day == 24).sum()
n_nulls = df_raw[numeric_cols].isnull().sum().sum()
n_dupes = df_raw.duplicated(subset=["date", "hour_ending"], keep=False).sum()
n_missing_dates = len(missing_dates)
temp_min, temp_max = df_raw["temp"].min(), df_raw["temp"].max()

checks = [
    ("Date range covers EXTENDED_FEATURE_START (2014)",
     df_raw["date"].min().date() <= pd.Timestamp("2014-01-01").date(),
     f"{df_raw['date'].min().date()} to {df_raw['date'].max().date()}"),
    ("All dates have 24 hours (excl. today)",
     n_complete >= n_dates - 1,
     f"{n_complete}/{n_dates} complete"),
    ("No missing calendar dates",
     n_missing_dates == 0,
     f"{n_missing_dates} missing"),
    ("No duplicate (date, hour) rows",
     n_dupes == 0,
     f"{n_dupes} duplicates"),
    ("Zero null values across all columns",
     n_nulls == 0,
     f"{n_nulls} total nulls"),
    ("Temperature within physical bounds (-30°F to 120°F)",
     temp_min >= -30 and temp_max <= 120,
     f"Range: [{temp_min}°F, {temp_max}°F]"),
    ("HDD formula verified",
     df_check["hdd_match"].all(),
     f"All {len(df_check):,} days match"),
    ("CDD formula verified",
     df_check["cdd_match"].all(),
     f"All {len(df_check):,} days match"),
    ("No days with both HDD>0 and CDD>0",
     df_check["both_positive"].sum() == 0,
     f"{df_check['both_positive'].sum()} violations"),
    ("HDD positively correlated with load",
     corr_hdd > 0.3,
     f"r = {corr_hdd:.3f}"),
    ("CDD positively correlated with load",
     corr_cdd > 0.3,
     f"r = {corr_cdd:.3f}"),
]

print("=" * 80)
print("  WEATHER DATA VALIDATION SUMMARY")
print("=" * 80)
all_pass = True
for name, passed, detail in checks:
    status = "PASS" if passed else "FAIL"
    if not passed:
        all_pass = False
    print(f"  [{status}] {name}")
    print(f"         {detail}")
print("=" * 80)
print(f"  {'ALL CHECKS PASSED' if all_pass else 'SOME CHECKS FAILED'}")
print("=" * 80)